In [ ]:
"""
Feature engineering: raw fastf1_pull.py telemetry -> TFT-ready feature table.

Transforms irregular per-lap telemetry into a fixed-rate (10Hz) grid with:
  - a strictly-increasing integer time_idx per lap (required by pytorch-forecasting)
  - remaining_lap_time as the target
  - the static / known-future / observed-past covariates TFT expects

See train_tft.py's module docstring for the exact target schema this produces.

Run:
    python build_features.py
"""

import logging
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("build_features")

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
RAW_DATA_DIR = Path("data/raw")
RESAMPLE_HZ = 5  # lowered from 10 -- still captures braking/throttle transitions
                  # well enough for lap-time prediction, at roughly half the row
                  # count and therefore roughly half the training time

# DRS codes >= 10 mean the driver actually has DRS open, not just eligible.
# See FastF1 docs -- 0/1 = off/unavailable, 8 = eligible-not-active,
# 10/12/14 = actually deployed. We only care about "is it open right now".
DRS_ACTIVE_CODES = {10, 12, 14}


OUTPUT_DIR = Path("data/features_5Hz")  # one parquet per session, not one giant file


def list_raw_files():
    files = sorted(RAW_DATA_DIR.glob("year=*/round=*/R.parquet"))
    if not files:
        raise FileNotFoundError(f"No raw parquet files found under {RAW_DATA_DIR}")
    logger.info(f"Found {len(files)} raw session files")
    return files


def compute_circuit_reference_distances(files) -> pd.Series:
    """
    Pass 1: figure out each circuit's typical lap distance WITHOUT loading full
    telemetry into memory for every session at once. We only read the handful
    of columns needed for this (Distance, driver/lap ids, EventName), compute
    a small per-lap summary, then immediately discard the rest of that file's
    data before moving to the next one.
    """
    needed_cols = ["Date", "Distance", "DriverNumber_x", "DriverNumber", "LapNumber", "EventName"]
    lap_spans = []

    for f in tqdm(files, desc="Pass 1/2: computing circuit reference distances"):
        # not every file necessarily has DriverNumber_x -- read what's actually there
        available = pd.read_parquet(f, columns=None).columns  # cheap: just reads schema/metadata-ish
        cols = [c for c in needed_cols if c in available]
        df = pd.read_parquet(f, columns=cols)
        df = fix_driver_number_column(df)
        df = df.dropna(subset=["LapNumber"])

        span = (
            df.groupby(["DriverNumber", "LapNumber"])["Distance"]
            .agg(lambda x: x.max() - x.min())
            .reset_index(name="lap_distance")
        )
        event_lookup = df[["DriverNumber", "LapNumber", "EventName"]].drop_duplicates()
        span = span.merge(event_lookup, on=["DriverNumber", "LapNumber"])
        lap_spans.append(span[["EventName", "lap_distance"]])

        del df  # explicit, since these frames can still be sizeable per file

    all_spans = pd.concat(lap_spans, ignore_index=True)
    circuit_distance = all_spans.groupby("EventName")["lap_distance"].median()
    logger.info(f"Computed reference lap distance for {len(circuit_distance)} circuits")
    return circuit_distance


def process_one_session_file(f: Path, circuit_distance: pd.Series) -> pd.DataFrame | None:
    """
    Full pipeline for ONE session's raw file. Kept fully self-contained so
    memory is released (via normal garbage collection) as soon as we move to
    the next file -- nothing here depends on any other session's telemetry.
    """
    df = pd.read_parquet(f)

    df = fix_driver_number_column(df)
    df = add_session_and_season(df)
    df = compute_drs_active(df)
    df = compute_brake_pct(df)
    df = filter_valid_laps(df)
    if df.empty:
        return None
    df = compute_stint_features(df)

    df["circuit_lap_distance"] = df["EventName"].map(circuit_distance)

    feature_df = process_all_laps(df)
    return feature_df


def fix_driver_number_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    fastf1_pull.py's merge produces DriverNumber_x/DriverNumber_y (both source
    frames had a DriverNumber column pre-merge) -- coalesce into one column.
    Worth fixing at the source eventually, but handled defensively here too.
    """
    if "DriverNumber_x" in df.columns:
        df["DriverNumber"] = df["DriverNumber_x"]
        df = df.drop(columns=[c for c in ["DriverNumber_x", "DriverNumber_y"] if c in df.columns])
    return df


def add_session_and_season(df: pd.DataFrame) -> pd.DataFrame:
    df["season"] = df["Year"].astype(int)
    df["session_id"] = (
        df["Year"].astype(str) + "_R" + df["RoundNumber"].astype(int).astype(str).str.zfill(2)
    )
    return df


def compute_drs_active(df: pd.DataFrame) -> pd.DataFrame:
    df["drs"] = df["DRS"].isin(DRS_ACTIVE_CODES).astype(float)
    return df


def compute_brake_pct(df: pd.DataFrame) -> pd.DataFrame:
    # FastF1's Brake is boolean (on/off), not a continuous pressure reading --
    # cast to float so it behaves as a normal numeric feature to the model.
    df["brake_pct"] = df["Brake"].astype(float)
    return df


def filter_valid_laps(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)

    # orphaned telemetry from the merge_asof tolerance cutoff -- no reliable lap to attribute to
    df = df.dropna(subset=["LapNumber"])

    # can't construct a target without a known final lap time
    df = df.dropna(subset=["LapTime"])

    # deleted laps (track limits, etc.) have a recorded time that doesn't
    # correspond to an accepted result -- exclude as a target
    if "Deleted" in df.columns:
        df = df[df["Deleted"] != True]  # noqa: E712 (explicit bool compare intentional here)

    logger.info(f"filter_valid_laps: {before:,} -> {len(df):,} rows ({before - len(df):,} dropped)")
    return df


def compute_stint_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    tyre_age_at_stint_start / compound_at_stint_start: constant across a whole
    stint, taken from that stint's first lap rather than recomputed per-lap.
    """
    stint_group = ["session_id", "DriverNumber", "Stint"]

    df = df.sort_values(["session_id", "DriverNumber", "LapNumber"])
    first_per_stint = df.groupby(stint_group, as_index=False).first()[
        stint_group + ["TyreLife", "Compound"]
    ].rename(columns={"TyreLife": "tyre_age_at_stint_start", "Compound": "compound_at_stint_start"})

    df = df.merge(first_per_stint, on=stint_group, how="left")
    return df


def resample_lap_to_grid(lap_df: pd.DataFrame, hz: int = RESAMPLE_HZ) -> pd.DataFrame | None:
    """
    Interpolate one lap's irregular telemetry onto a fixed-rate time grid.
    Returns None if the lap is too short/malformed to resample meaningfully.
    """
    lap_df = lap_df.sort_values("Date")
    if len(lap_df) < 5:
        return None

    t0 = lap_df["Date"].iloc[0]
    elapsed = (lap_df["Date"] - t0).dt.total_seconds().values

    lap_time_seconds = lap_df["LapTime"].iloc[0].total_seconds()
    if lap_time_seconds <= 0 or np.isnan(lap_time_seconds):
        return None

    dt = 1.0 / hz
    grid = np.arange(0, lap_time_seconds, dt)
    if len(grid) < 2:
        return None

    distance_into_lap_raw = (lap_df["Distance"] - lap_df["Distance"].iloc[0]).values

    numeric_cols = {
        "Speed": "speed",
        "Throttle": "throttle",
        "brake_pct": "brake_pct",
        "nGear": "gear",
        "drs": "drs",
        "AirTemp": "air_temp",
        "TrackTemp": "track_temp",
        "Humidity": "humidity",
        "WindSpeed": "wind_speed",
        "TyreLife": "tyre_life",
    }

    out = {"time_idx": np.arange(len(grid))}
    out["distance_into_lap"] = np.interp(grid, elapsed, distance_into_lap_raw)

    circuit_lap_distance = lap_df["circuit_lap_distance"].iloc[0]
    out["remaining_distance"] = np.clip(circuit_lap_distance - out["distance_into_lap"], 0, None)

    for src_col, out_col in numeric_cols.items():
        if src_col in lap_df.columns:
            out[out_col] = np.interp(grid, elapsed, lap_df[src_col].astype(float).values)

    out["remaining_lap_time"] = lap_time_seconds - grid

    result = pd.DataFrame(out)

    # static columns: same value for every row of this lap
    static_cols = {
        "session_id": lap_df["session_id"].iloc[0],
        "driver": lap_df["DriverNumber"].iloc[0],
        "lap_number": lap_df["LapNumber"].iloc[0],
        "season": lap_df["season"].iloc[0],
        "team": lap_df["Team"].iloc[0],
        "circuit": lap_df["EventName"].iloc[0],
        "event_format": lap_df["EventFormat"].iloc[0] if "EventFormat" in lap_df.columns else None,
        "compound_at_stint_start": lap_df["compound_at_stint_start"].iloc[0],
        "tyre_age_at_stint_start": lap_df["tyre_age_at_stint_start"].iloc[0],
    }
    for col, val in static_cols.items():
        result[col] = val

    return result


def process_all_laps(df: pd.DataFrame) -> pd.DataFrame:
    grouped = df.groupby(["session_id", "DriverNumber", "LapNumber"])
    results = []
    skipped = 0

    for _, lap_df in tqdm(grouped, desc="Resampling laps to fixed grid"):
        resampled = resample_lap_to_grid(lap_df)
        if resampled is None:
            skipped += 1
            continue
        results.append(resampled)

    logger.info(f"Resampled {len(results):,} laps successfully, skipped {skipped:,} (too short/malformed)")

    if not results:
        raise RuntimeError("No laps survived resampling -- check upstream filtering and data quality")

    return pd.concat(results, ignore_index=True)


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    files = list_raw_files()

    logger.info("Pass 1/2: computing per-circuit reference lap distances...")
    circuit_distance = compute_circuit_reference_distances(files)

    logger.info("Pass 2/2: processing each session and writing features incrementally...")
    total_rows = 0
    for f in tqdm(files, desc="Pass 2/2: building features"):
        # e.g. data/raw/year=2022/round=1/R.parquet -> data/features/year=2022_round=1.parquet
        out_name = f"{f.parent.parent.name}_{f.parent.name}.parquet"
        out_path = OUTPUT_DIR / out_name

        if out_path.exists():
            logger.info(f"Skipping {out_path} (already exists)")
            continue

        try:
            feature_df = process_one_session_file(f, circuit_distance)
        except Exception:
            logger.exception(f"Failed to process {f}, skipping")
            continue

        if feature_df is None or feature_df.empty:
            logger.warning(f"No valid laps produced for {f}")
            continue

        feature_df.to_parquet(out_path, index=False)
        total_rows += len(feature_df)
        logger.info(f"Saved {out_path} ({len(feature_df):,} rows)")

    logger.info(f"Done. {total_rows:,} total rows written across all sessions to {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

2026-09-14 10:49:33,013 [INFO] Found 70 raw session files
2026-09-14 10:49:33,020 [INFO] Pass 1/2: computing per-circuit reference lap distances...
Pass 1/2: computing circuit reference distances:   0%|          | 0/70 [00:00<?, ?it/s]

Pass 1/2: computing circuit reference distances: 100%|██████████| 70/70 [01:10<00:00,  1.01s/it]
2026-09-14 10:50:43,884 [INFO] Computed reference lap distance for 24 circuits
2026-09-14 10:50:43,888 [INFO] Pass 2/2: processing each session and writing features incrementally...
Pass 2/2: building features:   0%|          | 0/70 [00:00<?, ?it/s]2026-09-14 10:50:43,897 [INFO] Skipping data\features_5Hz\year=2023_round=1.parquet (already exists)
2026-09-14 10:50:43,897 [INFO] Skipping data\features_5Hz\year=2023_round=10.parquet (already exists)
2026-09-14 10:50:43,903 [INFO] Skipping data\features_5Hz\year=2023_round=11.parquet (already exists)
2026-09-14 10:50:43,906 [INFO] Skipping data\features_5Hz\year=2023_round=12.parquet (already exists)
2026-09-14 10:50:43,908 [INFO] Skipping data\features_5Hz\year=2023_round=13.parquet (already exists)
2026-09-14 10:50:43,909 [INFO] Skipping data\features_5Hz\year=2023_round=14.parquet (already exists)
2026-09-14 10:50:43,909 [INFO] Skipping dat